# Module Load — Chargement dans le warehouse PostgreSQL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haja171106/donnee2-aqi/blob/feat/notebooks-analysis/notebooks/load.ipynb)

Ce notebook analyse et documente le module `src/load.py`.

**Rôle :** Lire `data/clean/qualite_air.csv` et synchroniser les données dans le warehouse PostgreSQL (Neon) selon un schéma en étoile.

**Schéma cible :**
- `dim_ville` : nom, pays, latitude, longitude
- `dim_temps` : timestamp, date, heure, jour_semaine, is_weekend, mois, trimestre, annee
- `fact_qualite_air` : clés étrangères + AQI + 8 polluants

## 0. Configuration Colab

Cette cellule configure l'environnement que vous soyez dans Colab ou en local.

In [ ]:
import sys, os
from datetime import datetime
from pathlib import Path

import pandas as pd
import psycopg2
import psycopg2.extras

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    CLEAN_DIR = Path("/content/sample_data/clean")
    CLEAN_FILE = CLEAN_DIR / "qualite_air.csv"
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)

    import urllib.request
    print("Téléchargement du CSV depuis GitHub...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/haja171106/donnee2-aqi/feat/notebooks-analysis/data/clean/qualite_air.csv",
        CLEAN_FILE
    )
    print("CSV téléchargé ✓")

    from google.colab import userdata
    DATABASE_URL = userdata.get("DATABASE_URL", "")
    if not DATABASE_URL:
        print("⚠ DATABASE_URL non configurée — définissez-la dans les secrets Colab (icône 🔑) pour exécuter le chargement")
else:
    from config import CLEAN_FILE, DATABASE_URL
    print("Configuration locale ✓")

COMPONENT_COLS = ["co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3"]
JOURS_FR = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
print("Configuration terminée ✓")

## 1. Analyse des fonctions

### `_connect()`

Établit la connexion au warehouse PostgreSQL via `psycopg2` en utilisant l'URL de connexion stockée dans la variable d'environnement `DATABASE_URL`.

In [ ]:
def _connect():
    return psycopg2.connect(DATABASE_URL)

print("Fonction _connect définie ✓")
if IN_COLAB and not DATABASE_URL:
    print("⚠ Ajoutez DATABASE_URL dans les secrets Colab pour utiliser cette fonction")

---
### `_load_villes(conn, df) -> dict`

**Étape 1 — Dimension ville.**

Extrait les villes uniques du DataFrame, les insère ou met à jour dans `dim_ville` via `INSERT ... ON CONFLICT`, et retourne un dictionnaire `{(nom, pays): id_ville}`.

La clause `ON CONFLICT` rend l'opération **idempotente** : si une ville existe déjà, ses coordonnées sont mises à jour au lieu de créer un doublon.

In [ ]:
def _load_villes(conn, df: pd.DataFrame) -> dict:
    villes = df[["ville", "pays", "latitude", "longitude"]].drop_duplicates()
    rows = list(villes.itertuples(index=False, name=None))

    sql = """
        INSERT INTO dim_ville (nom, pays, latitude, longitude)
        VALUES %s
        ON CONFLICT (nom, pays) DO UPDATE
            SET latitude = EXCLUDED.latitude,
                longitude = EXCLUDED.longitude
        RETURNING id_ville, nom, pays
    """
    with conn.cursor() as cur:
        psycopg2.extras.execute_values(cur, sql, rows)
        result = cur.fetchall()

    with conn.cursor() as cur:
        cur.execute("SELECT id_ville, nom, pays FROM dim_ville")
        result = cur.fetchall()

    return {(nom, pays): id_ville for id_ville, nom, pays in result}

print("Fonction _load_villes définie ✓")

---
### `_load_temps(conn, df) -> dict`

**Étape 2 — Dimension temps.**

Pour chaque timestamp unique dans le CSV, calcule et insère les attributs temporels dans `dim_temps` :

| Attribut | Calcul |
|---|---|
| `timestamp_utc` | Valeur brute |
| `date` | `ts.date()` |
| `heure` | `ts.hour` |
| `jour_semaine` | `JOURS_FR[ts.weekday()]` |
| `is_weekend` | `ts.weekday() >= 5` |
| `mois` | `ts.month` |
| `trimestre` | `(ts.month - 1) // 3 + 1` |
| `annee` | `ts.year` |

Retourne un dictionnaire `{timestamp_iso: id_temps}`.

In [ ]:
def _load_temps(conn, df: pd.DataFrame) -> dict:
    timestamps = pd.to_datetime(df["timestamp_utc"], utc=True).drop_duplicates()

    rows = []
    for ts in timestamps:
        rows.append((
            ts.to_pydatetime(),
            ts.date(),
            ts.hour,
            JOURS_FR[ts.weekday()],
            ts.weekday() >= 5,
            ts.month,
            (ts.month - 1) // 3 + 1,
            ts.year,
        ))

    sql = """
        INSERT INTO dim_temps
            (timestamp_utc, date, heure, jour_semaine, is_weekend, mois, trimestre, annee)
        VALUES %s
        ON CONFLICT (timestamp_utc) DO NOTHING
    """
    with conn.cursor() as cur:
        psycopg2.extras.execute_values(cur, sql, rows)

    with conn.cursor() as cur:
        cur.execute("SELECT id_temps, timestamp_utc FROM dim_temps")
        result = cur.fetchall()

    return {ts.isoformat(): id_temps for id_temps, ts in result}

print("Fonction _load_temps définie ✓")

---
### `_load_faits(conn, df, ville_map, temps_map) -> int`

**Étape 3 — Table de faits.**

Associe chaque ligne du CSV aux clés étrangères (`id_ville`, `id_temps`) via les dictionnaires créés précédemment, puis insère ou met à jour la mesure dans `fact_qualite_air`.

**SQL :** `INSERT ... ON CONFLICT (id_temps, id_ville) DO UPDATE`

Cela permet de **rejouer** le chargement sans jamais créer de doublons : si une ligne existe déjà pour ce couple (ville, heure), les valeurs sont mises à jour.

In [ ]:
def _load_faits(conn, df: pd.DataFrame, ville_map: dict, temps_map: dict) -> int:
    rows = []
    for row in df.itertuples(index=False):
        id_ville = ville_map[(row.ville, row.pays)]
        ts_key = pd.to_datetime(row.timestamp_utc, utc=True).isoformat()
        id_temps = temps_map[ts_key]
        rows.append((
            id_temps, id_ville, row.aqi,
            row.co, row.no, row.no2, row.o3, row.so2, row.pm2_5, row.pm10, row.nh3,
        ))

    sql = """
        INSERT INTO fact_qualite_air
            (id_temps, id_ville, aqi, co, no, no2, o3, so2, pm2_5, pm10, nh3)
        VALUES %s
        ON CONFLICT (id_temps, id_ville) DO UPDATE
            SET aqi = EXCLUDED.aqi,
                co = EXCLUDED.co,
                no = EXCLUDED.no,
                no2 = EXCLUDED.no2,
                o3 = EXCLUDED.o3,
                so2 = EXCLUDED.so2,
                pm2_5 = EXCLUDED.pm2_5,
                pm10 = EXCLUDED.pm10,
                nh3 = EXCLUDED.nh3
    """
    with conn.cursor() as cur:
        psycopg2.extras.execute_values(cur, sql, rows)

    return len(rows)

print("Fonction _load_faits définie ✓")

---
### `load_warehouse()`

Fonction principale qui orchestre le chargement complet :
1. Lit `data/clean/qualite_air.csv`
2. Vérifie que le fichier n'est pas vide
3. Se connecte à la base
4. Charge les villes (`_load_villes`)
5. Charge les temps (`_load_temps`)
6. Charge les faits (`_load_faits`)
7. **Commit** la transaction (ou **rollback** en cas d'erreur)
8. Ferme la connexion

**Transactionnel :** tout ou rien — si une étape échoue, aucune modification n'est persistée.

In [ ]:
def load_warehouse() -> None:
    df = pd.read_csv(CLEAN_FILE)
    if df.empty:
        print("[load] CSV vide, rien à charger.")
        return

    conn = _connect()
    try:
        ville_map = _load_villes(conn, df)
        temps_map = _load_temps(conn, df)
        n = _load_faits(conn, df, ville_map, temps_map)
        conn.commit()
        print(f"[load] {len(ville_map)} villes, {len(temps_map)} horodatages, "
              f"{n} lignes de faits synchronisées")
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

print("Fonction load_warehouse définie ✓")

## 4. Analyse du schéma cible

Le warehouse suit un **schéma en étoile** avec 3 tables.

In [ ]:
if IN_COLAB:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/haja171106/donnee2-aqi/feat/notebooks-analysis/sql/schema.sql",
        "/content/schema.sql"
    )
    with open("/content/schema.sql") as f:
        schema = f.read()
else:
    with open("sql/schema.sql") as f:
        schema = f.read()

print("=== Schéma SQL ===")
print(schema)

## 5. Simulation du chargement (sans connexion)

Voyons comment les données seraient mappées vers le warehouse.

In [ ]:
df = pd.read_csv(CLEAN_FILE)

villes = df[["ville", "pays", "latitude", "longitude"]].drop_duplicates()
print(f"Villes qui seraient chargées dans dim_ville :")
villes

In [ ]:
JOURS_FR = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
timestamps = pd.to_datetime(df["timestamp_utc"], utc=True).drop_duplicates()
temps_rows = []
for ts in list(timestamps)[:5]:
    temps_rows.append({
        "timestamp_utc": ts.isoformat(),
        "date": str(ts.date()),
        "heure": ts.hour,
        "jour_semaine": JOURS_FR[ts.weekday()],
        "is_weekend": ts.weekday() >= 5,
        "mois": ts.month,
        "trimestre": (ts.month - 1) // 3 + 1,
        "annee": ts.year,
    })
pd.DataFrame(temps_rows)

## 6. Résumé

| Fonction | Rôle | Table cible |
|---|---|---|
| `_connect()` | Connexion PostgreSQL | — |
| `_load_villes()` | Upsert des villes | `dim_ville` |
| `_load_temps()` | Upsert des timestamps avec attributs dérivés | `dim_temps` |
| `_load_faits()` | Upsert des mesures | `fact_qualite_air` |
| `load_warehouse()` | Orchestration complète | Toutes |

**Points clés :**
- Requêtes **idempotentes** grâce à `INSERT ... ON CONFLICT`
- **Transactionnel** : commit global ou rollback intégral
- Schéma en **étoile** : `dim_ville` + `dim_temps` → `fact_qualite_air`
- Attributs temporels dérivés calculés automatiquement
- Pas de colonnes descriptives dans la table de faits